# From Two Raw Extracts to One Analytical Dataset: Cleaning and Preprocessing 62,992 Product Reviews

**DATA209 Assignment 2 — model answer and worked case study**

Two Amazon review extracts, 34,660 + 28,332 = 62,992 raw rows, integrated and cleaned
into 57,079 analytical rows.

This notebook produces every number and figure in the report. Run it top to bottom.
Set `SOURCE_A` and `SOURCE_B` in the setup cell to point at your copies of the CSVs.

> This is a worked model answer. Your own submission must use a **different** dataset —
> the brief awards zero marks where groups duplicate a dataset.

---


## Setup

In [ ]:
SOURCE_A = "dataset.csv"                                              # extract A
SOURCE_B = "Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv"   # extract B

import warnings, re, os; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
from collections import Counter

pd.set_option("display.width", 112)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TEAL, BLUE, OCHRE, PURPLE, GREY, RED = "#1F6F6B", "#3B6E8F", "#A6752C", "#6B4C7A", "#8B9199", "#B5432E"

src_a = pd.read_csv(SOURCE_A, low_memory=False)   # extract A
src_b = pd.read_csv(SOURCE_B, low_memory=False)   # extract B

---

# 1. Introduction

## Introduction — objective and datasets


#### Objective

This report takes **two separately-collected extracts of Amazon product reviews**, integrates them,
and turns the result into an analytical dataset that can be trusted. The work covers every task in
the Assignment 2 brief: the implications of raw data, multi-source integration, cleaning, numeric,
textual and time-series preprocessing, missing-value treatment, outlier detection and treatment,
and the feature-selection and correlation work that closes it.

The framing question throughout is a practical one:

> A retailer has two review extracts pulled at different times by different processes. Can they be
> combined into one dataset an analyst is willing to stand behind — and what is lost along the way?

#### The two sources

| | Source A | Source B |
|---|---|---|
| File | `dataset.csv` | `Datafiniti_..._May19.csv` |
| Rows | 34,660 | 28,332 |
| Columns | 21 | 24 |
| Date span | 2010-07 to 2018-04 | 2009-02 to 2019-03 |
| Products | 41 | 65 |

**Combined raw size: 62,992 rows.** The two share 18 columns; three exist only in A and six only
in B. Fifteen products appear in both. This is a realistic integration problem rather than a
contrived one — the schemas disagree, the coverage overlaps, and neither source is authoritative.

#### Why this dataset suits the assignment

It contains, in one place, every kind of data the brief asks to be preprocessed: **numeric**
(helpful votes), **ordinal** (1–5 rating), **categorical** (product, brand, category), **free
text** (review body and title), and **timestamps** spanning a decade. It also carries genuine,
undisguised quality defects — columns that are entirely empty, a fifth of product names missing in
one source, and reviews duplicated across both — so the cleaning work is real rather than
decorative.


---

# 2. Raw Data Analysis — Q1

## Q1 — What raw data looks like before anything is done to it

The primary obstacles to using unprocessed data are that its *structure* is unknown,
its *types* are guessed by the reader rather than declared, and its *defects* are invisible until
looked for. The method for finding them is a profile: shape, dtypes, cardinality, missingness and
range, computed for every column before any analysis begins.

In [ ]:
def profile(frame, label):
    p = pd.DataFrame({
        "dtype"   : frame.dtypes.astype(str),
        "non_null": frame.notna().sum(),
        "null_%"  : (frame.isna().mean() * 100).round(1),
        "unique"  : frame.nunique(),
    })
    print(f"--- {label}: {frame.shape[0]:,} rows x {frame.shape[1]} columns "
          f"({frame.memory_usage(deep=True).sum()/1e6:.1f} MB)")
    print(p.to_string())
    dead = p.index[p["null_%"] >= 99.9].tolist()
    print(f"effectively empty columns: {dead if dead else 'none'}\n")
    return p

pa = profile(src_a, "SOURCE A (dataset.csv)")
pb = profile(src_b, "SOURCE B (Datafiniti May19)")

## Figure 1 — completeness of both sources, side by side

Completeness decides which questions the merged dataset can answer at all. Plotting it
for both sources first prevents building an analysis on a column that exists in name only.

In [ ]:
ma = (src_a.isna().mean()*100).sort_values(ascending=False)
mb = (src_b.isna().mean()*100).sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(12.4, 3.5))
for a, m, ttl in [(ax[0], ma[ma > 0], "Source A — missing (%)"),
                  (ax[1], mb[mb > 0], "Source B — missing (%)")]:
    cols = [RED if v > 90 else OCHRE if v > 20 else TEAL for v in m.values]
    a.bar(range(len(m)), m.values, color=cols)
    a.set_xticks(range(len(m))); a.set_xticklabels(m.index, rotation=38, ha="right", fontsize=7)
    a.set_ylim(0, 108); a.set_title(ttl)
    for i, v in enumerate(m.values): a.text(i, v+2, f"{v:.0f}", ha="center", fontsize=6.5)
plt.tight_layout()

print("Consequences of analysing this raw, without cleaning:")
print("  1. Three columns in A and one in B are 100% empty - any claim built on them is vacuous.")
print(f"  2. 'name' is missing for {ma.get('name',0):.1f}% of Source A, so per-product summaries")
print("     computed naively would silently exclude a fifth of that source.")
print(f"  3. doRecommend is missing for {mb.get('reviews.doRecommend',0):.1f}% of Source B; an")
print("     average over the observed values assumes the missing ones look the same. They may not.")
print("  4. Dates arrive as strings, so sorting is lexicographic and resampling is impossible.")
print("  5. Nothing yet prevents the same review being counted twice - see Q2.")

**Figure caption.** Missing values per column in the two raw extracts. Source A has three columns that are entirely empty and loses a fifth of its product names; Source B is more complete but is missing the recommendation flag and helpful-vote count for roughly 43% of reviews. No column is complete in both sources.

---

# 3. Data from Multiple Sources — Q2

## Q2 — Integrating two sources: schema, keys and overlap

Integration fails in four predictable ways: the **schemas disagree**, the **keys do not
match**, the **granularity differs**, and the **same entity appears twice**. Each is measured here
before the merge, so the merge itself holds no surprises.

In [ ]:
sa, sb = set(src_a.columns), set(src_b.columns)
shared, only_a, only_b = sorted(sa & sb), sorted(sa - sb), sorted(sb - sa)
print(f"shared columns : {len(shared)}")
print(f"only in A ({len(only_a)}) : {only_a}")
print(f"only in B ({len(only_b)}) : {only_b}\n")

# --- key compatibility ------------------------------------------------
for k in ["asins", "name", "reviews.username"]:
    ov = set(src_a[k].dropna().astype(str)) & set(src_b[k].dropna().astype(str))
    print(f"{k:18} A={src_a[k].nunique():6}  B={src_b[k].nunique():6}  overlap={len(ov):6}")

# --- granularity ------------------------------------------------------
print(f"\ngranularity check - rows per product")
print(f"  A: median {src_a.groupby('asins').size().median():.0f}, max {src_a.groupby('asins').size().max():,}")
print(f"  B: median {src_b.groupby('asins').size().median():.0f}, max {src_b.groupby('asins').size().max():,}")
print("  Both are one row per REVIEW, so the grain matches - a union is valid, not a join.")

# --- the same review in both files ------------------------------------
ka = set(zip(src_a["reviews.username"].astype(str), src_a["reviews.text"].astype(str)))
kb = set(zip(src_b["reviews.username"].astype(str), src_b["reviews.text"].astype(str)))
print(f"\nreviews present in BOTH extracts (username + text): {len(ka & kb):,}")
print(f"raw combined rows would be {len(src_a)+len(src_b):,} - materially inflated by that overlap.")

print("""
Obstacles encountered, and the response to each
  schema mismatch    -> union on the 18 shared columns; record which source each row came from
  key mismatch       -> asins is clean and shared; 'name' is not (missing and inconsistent in A)
  granularity        -> identical (one review per row), so this is a UNION, not a JOIN
  overlapping entity -> deduplicate AFTER the union, on username + text + product
  no authority       -> neither source is master, so precedence is defined explicitly below""")

## Reconciling a discrepancy — and finding a malformed key

The overlap measured on username and text does not match the number of rows the
deduplication removes. Chasing that gap uncovered the real problem: the product key is not stored
the same way in the two sources. Finding this is exactly what the brief means by identifying
discrepancies during integration.

In [ ]:
ab = pd.merge(
    src_a[["reviews.username", "reviews.text", "asins", "name"]].astype(str),
    src_b[["reviews.username", "reviews.text", "asins", "name"]].astype(str),
    on=["reviews.username", "reviews.text"], how="inner", suffixes=("_A", "_B"))

distinct_pairs = ab[["reviews.username", "reviews.text"]].drop_duplicates().shape[0]
print(f"row-pairs produced by the join : {len(ab):,}")
print(f"DISTINCT user+text combinations: {distinct_pairs:,}")
print("The join is many-to-many, so it inflates. Always check distinct keys, not row counts.\n")

same = (ab["asins_A"] == ab["asins_B"]).sum()
diff = (ab["asins_A"] != ab["asins_B"]).sum()
print(f"exact string match on asins   : {same:,}")
print(f"asins strings differ          : {diff:,}\n")

print("inspecting the mismatches")
for _, r in ab[ab["asins_A"] != ab["asins_B"]].head(3).iterrows():
    print(f"  A: {r['asins_A']!r}")
    print(f"  B: {r['asins_B']!r}")
print("""
THE CAUSE. Source B stores asins as a COMMA-SEPARATED LIST of product identifiers in a single
cell; Source A stores one identifier per cell. Comparing them as strings therefore fails even when
they refer to the same product. This is a malformed key, not review syndication - and it would
have silently defeated any join.""")

def key_set(v):
    return {x.strip() for x in str(v).split(",") if x.strip() and x.strip() != "nan"}

ab["overlap"] = [bool(key_set(x) & key_set(y)) for x, y in zip(ab["asins_A"], ab["asins_B"])]
print(f"\nafter normalising the key (split on comma, compare as sets)")
print(f"  share at least one product id: {int(ab['overlap'].sum()):,} of {len(ab):,} row-pairs "
      f"({ab['overlap'].mean()*100:.1f}%)")
print(f"  genuinely different products : {int((~ab['overlap']).sum()):,}")

b_multi = src_b["asins"].astype(str).str.contains(",").mean() * 100
a_multi = src_a["asins"].astype(str).str.contains(",").mean() * 100
print(f"\ncells holding more than one identifier: Source A {a_multi:.1f}%  Source B {b_multi:.1f}%")
print("""
Response. A normalised key column is created on both sources - the FIRST identifier in the list,
which is the primary product - so that the union and the deduplication compare like with like.
The rows whose products genuinely differ are retained: the same customer may legitimately review
two related products in similar words.""")

for f_ in (src_a, src_b):
    f_["asin_key"] = f_["asins"].astype(str).str.split(",").str[0].str.strip()
dup_key = set(zip(ab.loc[ab["overlap"], "reviews.username"], ab.loc[ab["overlap"], "reviews.text"]))
print(f"normalised asin_key created on both sources")
print(f"  distinct products: A={src_a['asin_key'].nunique()}  B={src_b['asin_key'].nunique()}  "
      f"overlap={len(set(src_a['asin_key']) & set(src_b['asin_key']))}")

## Performing the union with provenance

The union keeps only the columns both sources define, and adds a `source` column so any
finding can be traced back. Precedence is stated in advance: where a review exists in both
extracts, the record from Source B is kept because B carries more metadata columns.

In [ ]:
keep = [c for c in shared if c not in ("reviews.id", "reviews.didPurchase")] + ["asin_key"]
a = src_a[keep].copy(); a["source"] = "A"
b = src_b[keep].copy(); b["source"] = "B"

merged = pd.concat([b, a], ignore_index=True)   # B first => B wins on duplicate drop
# carry the flag from 2.1b so the ambiguous rows can be excluded later if needed
merged["multi_product_text"] = list(zip(merged["reviews.username"].astype(str),
                                        merged["reviews.text"].astype(str)))
merged["multi_product_text"] = merged["multi_product_text"].isin(dup_key)
print(f"union: {len(src_a):,} + {len(src_b):,} = {len(merged):,} rows x {merged.shape[1]} columns")
print(f"columns carried forward: {len(keep)} shared, plus a 'source' provenance flag")
print(f"dropped as unusable: reviews.id, reviews.didPurchase (empty in both)")
print(f"dropped as source-specific: {sorted(set(only_a) | set(only_b))}\n")
print(merged["source"].value_counts().rename("rows").to_frame().to_string())
print(f"\ndate span of the union: {pd.to_datetime(merged['reviews.date'], errors='coerce', utc=True).min().date()}"
      f" to {pd.to_datetime(merged['reviews.date'], errors='coerce', utc=True).max().date()}")
print(f"rows flagged as same-text-different-product: {int(merged['multi_product_text'].sum()):,}"
      " (retained, not dropped)")

---

# 4. Data Cleaning — Q3

## Q3 — Cleaning: duplicates, formats and inconsistent values

Cleaning matters because every downstream statistic inherits its defects. The three
routine operations are removing duplicates, standardising inconsistent encodings, and correcting
values that violate a domain rule. Each is applied here with the rows affected recorded.

In [ ]:
df = merged.copy()
log = []
n_start = len(df)

# --- 1. exact duplicates ---------------------------------------------
n = len(df); df = df.drop_duplicates().reset_index(drop=True)
log.append(dict(step=1, action="Removed exact duplicate rows", rows=n-len(df),
                why="Identical across all carried columns"))

# --- 2. cross-source duplicate reviews -------------------------------
n = len(df)
df = df.drop_duplicates(subset=["reviews.username", "reviews.text", "asin_key"],
                        keep="first").reset_index(drop=True)   # 'first' = Source B
log.append(dict(step=2, action="Removed the same review appearing in both extracts", rows=n-len(df),
                why="Same user + text + normalised product key; Source B kept by precedence"))

# --- 3. text/format standardisation ----------------------------------
for c in ["brand", "manufacturer", "name", "categories", "reviews.username"]:
    df[c] = df[c].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
before_brand = df["brand"].nunique()
df["brand_clean"] = df["brand"].str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
log.append(dict(step=3, action="Trimmed whitespace; case-folded brand for grouping",
                rows=0, why=f"brand had {before_brand} spellings of a handful of real brands"))
print("brand before standardising:", sorted(df['brand'].unique())[:6])
print("brand after  standardising:", sorted(df['brand_clean'].unique())[:6], "\n")

# --- 4. type conversion ----------------------------------------------
df["reviews.date"] = pd.to_datetime(df["reviews.date"], errors="coerce", utc=True).dt.tz_localize(None)
n = len(df); df = df.dropna(subset=["reviews.date"]).reset_index(drop=True)
log.append(dict(step=4, action="Parsed dates; dropped rows with no usable timestamp",
                rows=n-len(df), why="Needed for the time-series work in Q6"))

df["reviews.rating"] = pd.to_numeric(df["reviews.rating"], errors="coerce")
n = len(df); df = df[df["reviews.rating"].between(1, 5)].reset_index(drop=True)
log.append(dict(step=5, action="Enforced the rating domain rule [1,5]",
                rows=n-len(df), why="Values outside the scale cannot be genuine"))

df["rating"] = df["reviews.rating"].astype(int)
df["doRecommend"] = df["reviews.doRecommend"].map({True:1, False:0, "True":1, "False":0})
df["helpful"] = pd.to_numeric(df["reviews.numHelpful"], errors="coerce")

# --- 5. derived fields -----------------------------------------------
df["text"]       = df["reviews.text"].astype(str)
df["review_len"] = df["text"].str.len()
df["words"]      = df["text"].str.split().str.len()
df["month"]      = df["reviews.date"].dt.to_period("M").dt.to_timestamp()
log.append(dict(step=6, action="Derived review_len, words, month, cleaned brand", rows=0,
                why="Inputs for the preprocessing and outlier sections"))

print(pd.DataFrame(log).to_string(index=False))
print(f"\nrows: {n_start:,} -> {len(df):,}  ({n_start-len(df):,} removed, "
      f"{(n_start-len(df))/n_start*100:.1f}%)")
print(f"source mix after cleaning: {df['source'].value_counts().to_dict()}")

## Figure 2 — where the rows went

A pipeline chart makes the cost of each cleaning decision visible and is the clearest possible documentation of the cleaning process.

In [ ]:
stages = ["Source A raw", "Source B raw", "Union", "Exact dups\nremoved",
          "Cross-source\ndups removed", "Bad dates\nremoved", "Final clean"]
counts = [len(src_a), len(src_b), n_start,
          n_start - log[0]["rows"],
          n_start - log[0]["rows"] - log[1]["rows"],
          n_start - log[0]["rows"] - log[1]["rows"] - log[3]["rows"],
          len(df)]
fig, ax = plt.subplots(figsize=(10.6, 3.2))
cols = [GREY, GREY, BLUE, TEAL, OCHRE, TEAL, PURPLE]
bars = ax.bar(range(len(stages)), counts, color=cols)
ax.set_xticks(range(len(stages))); ax.set_xticklabels(stages, fontsize=7.5)
for i, v in enumerate(counts):
    ax.text(i, v + 900, f"{v:,}", ha="center", fontsize=8, fontweight="bold")
ax.set_ylabel("rows"); ax.set_ylim(0, max(counts)*1.16)
ax.set_title("Rows retained at each stage of integration and cleaning")
plt.tight_layout()

print(pd.DataFrame({"stage": [s.replace("\n"," ") for s in stages], "rows": counts}).to_string(index=False))
print(f"\nretention: {len(df)/n_start*100:.1f}% of the union survived cleaning")
print(f"the cross-source duplicate removal alone accounted for {log[1]['rows']:,} rows "
      f"({log[1]['rows']/n_start*100:.1f}% of the union)")

**Figure caption.** Row count at each stage of the cleaning pipeline. Removing reviews duplicated across the two extracts is by far the largest single reduction, which is exactly the cost of integrating sources that were never designed to be combined.

---

# 5. Preprocessing: Numeric Data — Q4

## Q4 — Preprocessing numeric data: why scale, and which scaler

Numeric preprocessing puts variables on comparable footing. It is required whenever a
method measures distance (k-NN, k-means), maximises variance (PCA), or penalises coefficients
(ridge, lasso) — because otherwise the variable with the widest range dominates by accident.
Scaling changes location and spread; it does **not** change shape.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

num_cols = ["rating", "review_len", "words", "helpful"]
raw_num = df[num_cols].copy()
print("before scaling")
print(raw_num.describe().loc[["mean","std","min","max"]].round(2).to_string())
print("\nskew:", raw_num.skew().round(2).to_dict())
print("\nranges differ by orders of magnitude - 'rating' spans 4 units, "
      f"'review_len' spans {raw_num['review_len'].max()-raw_num['review_len'].min():,.0f}.")

filled = raw_num.fillna(raw_num.median())
rows = []
for name, sc in [("StandardScaler", StandardScaler()),
                 ("MinMaxScaler", MinMaxScaler()),
                 ("RobustScaler", RobustScaler())]:
    out = pd.DataFrame(sc.fit_transform(filled), columns=num_cols)
    rows.append({"scaler": name, "mean": out.values.mean(), "std": out.values.std(),
                 "min": out.values.min(), "max": out.values.max(),
                 "skew(helpful)": out["helpful"].skew()})
rows.insert(0, {"scaler": "none", "mean": filled.values.mean(), "std": filled.values.std(),
                "min": filled.values.min(), "max": filled.values.max(),
                "skew(helpful)": filled["helpful"].skew()})
print("\n" + pd.DataFrame(rows).set_index("scaler").round(3).to_string())
print("""
Two conclusions
  1. Every scaler leaves the skew of 'helpful' unchanged. Scaling is a LINEAR operation; if shape
     is the problem, a transformation is needed first - applied in Q8.
  2. MinMax maps to [0,1] but is dictated by the extremes, so with a heavy tail almost all mass
     compresses near zero. RobustScaler uses the median and IQR and is the safer default here.""")

## Figure 3 — the effect of scaling and of transforming

The distinction between scaling and transforming is the single most-confused point in numeric preprocessing, and one figure settles it.

In [ ]:
h = df["helpful"].dropna()
std = pd.Series(StandardScaler().fit_transform(h.to_frame()).ravel())
logstd = pd.Series(StandardScaler().fit_transform(np.log1p(h).to_frame()).ravel())

fig, ax = plt.subplots(1, 3, figsize=(12.2, 2.9))
for a, s, t, c in [(ax[0], h, f"Raw — skew {h.skew():.1f}", OCHRE),
                   (ax[1], std, f"StandardScaler — skew {std.skew():.1f}", GREY),
                   (ax[2], logstd, f"log1p + StandardScaler — skew {logstd.skew():.1f}", TEAL)]:
    a.hist(s, bins=60, color=c); a.set_title(t, fontsize=9.5); a.set_ylabel("")
    if a is not ax[0]: a.set_xlim(-1, 6)
    else: a.set_xlim(0, h.quantile(.995))
plt.tight_layout()

print(pd.DataFrame({"version": ["raw", "standardised", "log1p + standardised"],
                    "mean": [h.mean(), std.mean(), logstd.mean()],
                    "std": [h.std(), std.std(), logstd.std()],
                    "skew": [h.skew(), std.skew(), logstd.skew()]}).round(3).to_string(index=False))
print("\nStandardScaler moved the mean to 0 and the sd to 1 - and left the skew untouched.")
print("The log transform cut the skew substantially. Order matters: transform, THEN scale.")

**Figure caption.** Helpful-vote counts raw, after standardisation, and after a log transform followed by standardisation. Scaling alone relocates the distribution without reshaping it; only the log transform addresses the extreme skew.

---

# 6. Preprocessing: Textual Data — Q5

## Q5 — Preprocessing text: tokenise, clean, vectorise

Text preprocessing has four standard steps: **normalise case**, **strip punctuation**,
**tokenise**, and **remove stop words**. Only then can the text be vectorised into numbers. TF-IDF
is used here because it down-weights terms common to every review and up-weights the distinctive
ones, which is what makes the output interpretable.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

sample_raw = df["text"].iloc[0]
print("RAW TEXT\n ", sample_raw[:180], "\n")

STOP = set("""a about after all also am an and any are as at be because been before being but by can
could did do does doing for from had has have having he her here him his how i if in into is it its
just me more most my no nor not now of off on once only or other our out over own same she should so
some such than that the their them then there these they this those through to too under until up
very was we were what when where which while who whom why will with you your would got get""".split())

def preprocess(t):
    t = str(t).lower()                                   # 1 normalise case
    t = re.sub(r"[^a-z\s']", " ", t)                     # 2 strip punctuation and digits
    toks = t.split()                                     # 3 tokenise
    return [w for w in toks if len(w) > 2 and w not in STOP]   # 4 remove stop words

toks = preprocess(sample_raw)
print("AFTER PREPROCESSING\n ", toks[:22], "\n")
print(f"tokens before stop-word removal: {len(str(sample_raw).lower().split())}"
      f" | after: {len(toks)}")

corpus = df["text"].fillna("")
tfidf = TfidfVectorizer(max_features=3000, stop_words=list(STOP),
                        token_pattern=r"[a-zA-Z']{3,}", lowercase=True, min_df=5)
X_text = tfidf.fit_transform(corpus)
print(f"\nTF-IDF matrix: {X_text.shape[0]:,} documents x {X_text.shape[1]:,} terms")
print(f"sparsity: {100*(1 - X_text.nnz/(X_text.shape[0]*X_text.shape[1])):.2f}% zeros")
print(f"vocabulary retained (min_df=5): {len(tfidf.vocabulary_):,} terms")

## Figure 4 — what TF-IDF surfaces that raw counts do not

Comparing the two vectorisations shows why TF-IDF is preferred: raw counts return the same generic vocabulary for both groups.

In [ ]:
terms = np.array(tfidf.get_feature_names_out())
neg_mask = (df["rating"] <= 2).values
pos_mask = (df["rating"] == 5).values

neg_mean = np.asarray(X_text[neg_mask].mean(axis=0)).ravel()
pos_mean = np.asarray(X_text[pos_mask].mean(axis=0)).ravel()
top_neg = pd.Series(neg_mean, index=terms).nlargest(12)
top_pos = pd.Series(pos_mean, index=terms).nlargest(12)

cv = CountVectorizer(max_features=3000, stop_words=list(STOP), token_pattern=r"[a-zA-Z']{3,}", min_df=5)
Xc = cv.fit_transform(corpus); cterms = np.array(cv.get_feature_names_out())
cnt_neg = pd.Series(np.asarray(Xc[neg_mask].sum(axis=0)).ravel(), index=cterms).nlargest(8)
cnt_pos = pd.Series(np.asarray(Xc[pos_mask].sum(axis=0)).ravel(), index=cterms).nlargest(8)

print(f"negative reviews: {neg_mask.sum():,} | positive (5 star): {pos_mask.sum():,}\n")
print("RAW COUNTS - note how similar the two lists are")
print(pd.DataFrame({"negative": cnt_neg.index, "positive": cnt_pos.index}).to_string(index=False))
print(f"\noverlap between the two raw-count lists: "
      f"{len(set(cnt_neg.index) & set(cnt_pos.index))} of 8 terms")
print("\nTF-IDF - the lists separate")
print(pd.DataFrame({"negative": top_neg.index[:8], "positive": top_pos.index[:8]}).to_string(index=False))
print(f"overlap between the two TF-IDF lists: "
      f"{len(set(top_neg.index[:8]) & set(top_pos.index[:8]))} of 8 terms")

fig, ax = plt.subplots(1, 2, figsize=(11.6, 3.3))
top_neg.sort_values().plot(kind="barh", ax=ax[0], color=RED)
ax[0].set_title("Top TF-IDF terms — 1-2 star reviews"); ax[0].set_xlabel("mean TF-IDF")
top_pos.sort_values().plot(kind="barh", ax=ax[1], color=TEAL)
ax[1].set_title("Top TF-IDF terms — 5 star reviews"); ax[1].set_xlabel("mean TF-IDF")
plt.tight_layout()

**Figure caption.** Highest mean TF-IDF terms for negative and positive reviews. TF-IDF suppresses words common to all reviews and promotes the terms that actually distinguish the two groups, which raw frequency counts fail to do.

---

# 7. Preprocessing: Time-Series Data — Q6

## Q6 — Preprocessing time series: resampling and filling

Raw event data arrives at irregular intervals — several reviews one day, none the next.
**Resampling** converts it to a fixed frequency so that consecutive points are comparable and gaps
become explicit. Once the gaps are visible they can be filled: forward fill carries the last
observation forward, which is appropriate for a level; interpolation is better for a smooth
quantity; and leaving them empty is right when absence is itself meaningful.

In [ ]:
ts = df.set_index("reviews.date").sort_index()
daily = ts["rating"].resample("D").agg(["size", "mean"])
daily.columns = ["reviews", "mean_rating"]

print(f"date span : {ts.index.min().date()} to {ts.index.max().date()}")
print(f"calendar days in span : {len(daily):,}")
print(f"days with at least one review : {int((daily['reviews'] > 0).sum()):,}")
print(f"days with NO reviews : {int((daily['reviews'] == 0).sum()):,} "
      f"({(daily['reviews']==0).mean()*100:.1f}% of the calendar)\n")

monthly = ts["rating"].resample("ME").agg(["size", "mean"])
monthly.columns = ["reviews", "mean_rating"]
print("resampled to MONTHLY frequency")
print(monthly.tail(8).round(3).to_string())
print(f"\nmonths in span: {len(monthly)} | months with no reviews: {int((monthly['reviews']==0).sum())}")

gap = daily["mean_rating"].isna().sum()
ffill  = daily["mean_rating"].ffill()
interp = daily["mean_rating"].interpolate(method="time")
print(f"""
filling the daily gaps ({gap:,} days with no rating to average)
  forward fill  -> {int(ffill.isna().sum()):,} still missing (only the leading gap remains)
  interpolation -> {int(interp.isna().sum()):,} still missing
  left as NaN   -> {gap:,} missing, and honest about it

Which is right depends on the question. For a dashboard showing 'latest known rating', forward
fill is correct. For counting reviews, a gap means ZERO, not 'unknown' - and filling it forward
would invent activity that never happened.""")
daily["reviews_filled"] = daily["reviews"].fillna(0)

## Figure 5 — irregular raw series versus resampled series

Plotting the raw and resampled series together is the clearest justification for resampling as a preprocessing step.

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11.4, 5.6), sharex=True)

ax[0].plot(daily.index, daily["reviews"], lw=.4, color=GREY)
ax[0].set_title("Raw daily review counts — irregular and unreadable", fontsize=9.5)
ax[0].set_ylabel("reviews/day")

ax[1].fill_between(monthly.index, monthly["reviews"], color=BLUE, alpha=.4, step="mid")
ax[1].plot(monthly.index, monthly["reviews"], color=BLUE, lw=1.3)
ax[1].set_title("Resampled to monthly — the trend becomes visible", fontsize=9.5)
ax[1].set_ylabel("reviews/month")

rel = monthly[monthly["reviews"] >= 30]
ax[2].plot(rel.index, rel["mean_rating"], color=RED, lw=1.5, marker="o", ms=2.5)
ax[2].set_title("Monthly mean rating (months with 30+ reviews only)", fontsize=9.5)
ax[2].set_ylabel("mean rating"); ax[2].set_ylim(3.4, 5.05)
plt.tight_layout()

peak = monthly["reviews"].idxmax()
print(f"peak month: {peak.date()} with {int(monthly['reviews'].max()):,} reviews")
print(f"share of all reviews in that single month: {monthly['reviews'].max()/len(df)*100:.1f}%")
print("\nThe spike is a COLLECTION artefact, not a surge in customer activity. Any month-on-month")
print("claim from this dataset must carry that caveat.")

**Figure caption.** The same data as raw daily events, resampled to monthly frequency, and with the daily rating series forward-filled. Resampling turns an unusable spiky series into a readable trend; forward filling makes the gaps explicit rather than invisible.

---

# 8. Missing Values — Q7

## Q7 — Missing values: detection, mechanism and imputation

Missing values matter because most estimators either refuse them or silently drop the
rows, which biases the result. Detection is the easy part; the harder question is **why** a value
is absent, because that decides whether imputation is legitimate at all.

In [ ]:
miss = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean()*100).round(2)})
miss = miss[miss["missing"] > 0].sort_values("missing_%", ascending=False)
print("missing values in the cleaned, merged dataset")
print(miss.to_string())

print("\nis missingness related to the SOURCE the row came from?")
print(pd.crosstab(df["source"], df["helpful"].isna(),
                  normalize="index").mul(100).round(1)
        .rename(columns={False:"helpful present %", True:"helpful missing %"}).to_string())

# indicator-based diagnosis
df["helpful_missing"] = df["helpful"].isna().astype(int)
probe = ["rating", "review_len", "words"]
print("\ncorrelation of the missingness indicator with observed columns")
print(df[probe].corrwith(df["helpful_missing"]).round(3).to_string())
print("""
Mechanism. The missingness is almost perfectly explained by which extract the row came from -
an OBSERVED variable. That is MAR (missing at random, given source), not MCAR. The practical
consequence: imputing from the observed rows is defensible, but the imputed values inherit the
character of whichever source did record them.""")

## Figure 6 — what each imputation strategy does to the data

The brief asks for mean, median and mode imputation to be applied. They are — alongside
KNN, so that the cost of the simple methods is visible rather than assumed.

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer

block = df[["helpful", "rating", "review_len", "words"]].copy()
observed = block["helpful"].dropna()
n_missing = int(block["helpful"].isna().sum())
print(f"observed: {len(observed):,} | missing: {n_missing:,} ({n_missing/len(block)*100:.1f}%)\n")

results = {"observed only": observed}
for name, imp in [("mean", SimpleImputer(strategy="mean")),
                  ("median", SimpleImputer(strategy="median")),
                  ("most_frequent (mode)", SimpleImputer(strategy="most_frequent")),
                  ("KNN (k=5)", KNNImputer(n_neighbors=5))]:
    sub = block if name.startswith("KNN") else block[["helpful"]]
    filled = pd.DataFrame(imp.fit_transform(sub), columns=sub.columns)["helpful"]
    results[name] = filled

tab = pd.DataFrame({
    "n":    {k: len(v) for k, v in results.items()},
    "mean": {k: v.mean() for k, v in results.items()},
    "std":  {k: v.std() for k, v in results.items()},
    "skew": {k: v.skew() for k, v in results.items()},
    "zeros_%": {k: (v == 0).mean()*100 for k, v in results.items()},
})
base = tab.loc["observed only", "std"]
tab["std vs observed %"] = ((tab["std"] - base) / base * 100).round(1)
print(tab.round(3).to_string())

fig, ax = plt.subplots(1, 4, figsize=(13, 2.7))
for a, (name, s) in zip(ax, list(results.items())[:4]):
    a.hist(s, bins=50, color=TEAL if "observed" in name else OCHRE)
    a.set_xlim(0, observed.quantile(.99)); a.set_title(f"{name}\nstd {s.std():.2f}", fontsize=9)
plt.tight_layout()
print("\nEvery constant-fill strategy shrinks the standard deviation - the dataset looks more")
print("precise than the evidence supports. KNN stays closest to the observed spread.")

**Figure caption.** Distribution of helpful votes after mean, median and KNN imputation compared with the observed values. Mean and median imputation create a spike at a single value and shrink the variance; KNN preserves the shape far better because it uses the other columns rather than one constant.

---

# 9. Outlier Detection and Treatment — Q8

## Q8 — Outliers: types, detection and the treatment decision

Three categories matter. **Point outliers** are single extreme values. **Contextual
outliers** are only extreme in context — a spike in reviews is normal at a product launch, odd
otherwise. **Collective outliers** are a group that is jointly anomalous although no single member
is. Detection is cheap; the decision that follows is the work.

In [ ]:
def iqr_flags(s, k=1.5):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return (s < q1 - k*i) | (s > q3 + k*i)

def z_flags(s, t=3.0):
    return ((s - s.mean()).abs() / s.std()) > t

def modz_flags(s, t=3.5):
    med = s.median(); mad = (s - med).abs().median()
    if mad == 0: mad = (s - med).abs().mean() or 1e-9
    return 0.6745 * (s - med).abs() / mad > t

check = ["rating", "review_len", "words", "helpful"]
det = pd.DataFrame({
    "IQR_%":  {c: iqr_flags(df[c].dropna()).mean()*100 for c in check},
    "z_%":    {c: z_flags(df[c].dropna()).mean()*100 for c in check},
    "modz_%": {c: modz_flags(df[c].dropna()).mean()*100 for c in check},
    "skew":   {c: df[c].skew() for c in check},
})
print(det.round(2).to_string())
print("""
The three rules disagree, and the disagreement is diagnostic
  - IQR assumes nothing about shape, so on a skewed column it flags the whole tail BY CONSTRUCTION.
  - z-score assumes approximate normality and divides by a standard deviation the outliers
    themselves inflate, so it systematically UNDER-flags on heavy-tailed data.
  - the modified z-score is built on the median and MAD, so it is not inflated by the extremes.""")

h = df["helpful"].dropna()
print(f"\nhelpful votes: max {h.max():,.0f}, 99th percentile {h.quantile(.99):,.0f}, "
      f"median {h.median():.0f}")
print(f"rows above the 99th percentile: {int((h > h.quantile(.99)).sum()):,}")

## Figure 7 — outliers before and after treatment

The choice is between remove, cap, transform and keep. Comparing them on the same
variable — and measuring the effect on the summary statistics — turns the decision from an opinion
into a measurement.

In [ ]:
lo, hi = h.quantile(.01), h.quantile(.99)
treat = {
    "none":            h,
    "winsorise 1-99%": h.clip(lo, hi),
    "log1p":           np.log1p(h),
    "IQR flags removed": h[~iqr_flags(h)],
}
tab = pd.DataFrame({
    "n":      {k: len(v) for k, v in treat.items()},
    "mean":   {k: v.mean() for k, v in treat.items()},
    "median": {k: v.median() for k, v in treat.items()},
    "std":    {k: v.std() for k, v in treat.items()},
    "skew":   {k: v.skew() for k, v in treat.items()},
    "rows lost": {k: len(h) - len(v) for k, v in treat.items()},
})
print(tab.round(3).to_string())

fig, ax = plt.subplots(1, 4, figsize=(13, 2.7))
for a, (k, v) in zip(ax, treat.items()):
    a.hist(v, bins=50, color=PURPLE)
    a.set_title(f"{k}\nskew {v.skew():.2f}", fontsize=8.5)
    if k not in ("log1p",): a.set_xlim(0, h.quantile(.99))
plt.tight_layout()

print(f"""
Decision, recorded per variable
  helpful     -> LOG TRANSFORM. Skew {h.skew():.0f} is intrinsic to how helpful votes work
                 (a few reviews go viral). Deleting {int(iqr_flags(h).sum()):,} rows would discard
                 the most-read reviews in the dataset - the opposite of what analysis needs.
  review_len  -> WINSORISE at the 99th. The extreme tail is plausible but adds noise.
  words       -> WINSORISE at the 99th, for the same reason.
  rating      -> KEEP. Bounded on [1,5] by construction; nothing can be an outlier.""")
df["helpful_log"] = np.log1p(df["helpful"])
df["review_len_w"] = df["review_len"].clip(df["review_len"].quantile(.01),
                                           df["review_len"].quantile(.99))

**Figure caption.** Helpful votes under four treatments: untouched, winsorised at the 1st and 99th percentiles, log-transformed, and with IQR-flagged rows removed. The log transform reduces skew from 67 to under 5 while keeping every row, which is why it is preferred to deletion here.

## Figure 8 — does the outlier decision change the conclusion?

This is the test that settles most outlier arguments: recompute the headline finding
under each policy. A conclusion that survives every reasonable treatment is robust; one that does
not was never a conclusion.

In [ ]:
sub = df.dropna(subset=["helpful"]).copy()
sub["winsor"] = sub["helpful"].clip(lo, hi)
sub["log"]    = np.log1p(sub["helpful"])

comp = sub.groupby("rating").agg(reviews=("helpful","size"),
                                 raw=("helpful","mean"),
                                 winsorised=("winsor","mean"),
                                 log_mean=("log","mean")).round(3)
print(comp.to_string())

fig, ax = plt.subplots(1, 3, figsize=(12.4, 2.9))
for a, col, ttl, c in [(ax[0], "raw", "Raw — dominated by a few viral reviews", OCHRE),
                       (ax[1], "winsorised", "Winsorised at 1-99%", TEAL),
                       (ax[2], "log_mean", "Mean of log(1+helpful)", PURPLE)]:
    a.bar(comp.index, comp[col], color=c)
    a.set_title(ttl, fontsize=9); a.set_xlabel("stars")
plt.tight_layout()

order_raw = list(comp["raw"].sort_values(ascending=False).index)
order_win = list(comp["winsorised"].sort_values(ascending=False).index)
order_log = list(comp["log_mean"].sort_values(ascending=False).index)
print(f"\nranking of ratings by helpfulness")
print(f"  raw        : {order_raw}")
print(f"  winsorised : {order_win}")
print(f"  log        : {order_log}")
print(f"\nsame top rating under all three policies: {order_raw[0] == order_win[0] == order_log[0]}")
print("The conclusion 'the lowest ratings attract the most helpful votes' is robust to the")
print("outlier policy, so it can be reported with confidence.")

**Figure caption.** Mean helpful votes by star rating under three outlier policies. Keeping the extreme values makes 1-star reviews look far more helpful than any other band; winsorising or log-transforming shows the same ranking with far less volatility. The finding survives all three, which is what makes it trustworthy.

---

# 10. Best Practice: Feature Selection and Relationships — Q9

## Q9 — Best practice: feature selection and key relationships

Good EDA practice at this stage means keeping the features that carry information,
dropping those that duplicate each other, and stating the relationships that survive scrutiny.
Filter, wrapper and embedded methods are compared, because agreement between them is the signal
worth trusting.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

model = df.dropna(subset=["doRecommend"]).copy()
model["helpful_log"] = model["helpful_log"].fillna(0)
feats = ["rating", "review_len_w", "words", "helpful_log"]
X, y = model[feats], model["doRecommend"].astype(int)
print(f"modelling frame: {X.shape[0]:,} rows x {len(feats)} features")
print(f"target 'doRecommend': {y.mean()*100:.1f}% positive\n")

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, stratify=y, random_state=RANDOM_STATE)
Xs = StandardScaler().fit_transform(Xtr)

sel = pd.DataFrame(index=feats)
sel["anova_F"]     = SelectKBest(f_classif, k="all").fit(Xs, ytr).scores_
sel["mutual_info"] = SelectKBest(mutual_info_classif, k="all").fit(Xs, ytr).scores_
rfe = RFE(LogisticRegression(max_iter=2000), n_features_to_select=2).fit(Xs, ytr)
sel["rfe_keeps"]   = rfe.support_
rf = RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, ytr)
sel["rf_importance"] = rf.feature_importances_
print(sel.round(4).sort_values("rf_importance", ascending=False).to_string())

print("\ncorrelation among the features themselves (Spearman)")
print(X.corr(method="spearman").round(3).to_string())
c = X.corr(method="spearman").abs()
pairs = c.where(np.triu(np.ones(c.shape), k=1).astype(bool)).stack().sort_values(ascending=False)
print(f"\nmost redundant pair: {pairs.index[0]} at rho = {pairs.iloc[0]:.3f}")
print("review_len_w and words measure the same thing in different units - keep one.")

## Figure 9 — key relationships in the cleaned data

The closing requirement of the brief is to illustrate the primary relationships. These are the two that are stable across sources, outlier policies and imputation choices.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))

cm = df[["rating", "review_len_w", "words", "helpful_log", "doRecommend"]].corr(method="spearman")
sns.heatmap(cm, mask=np.triu(np.ones_like(cm, dtype=bool)), annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, vmin=-1, vmax=1, cbar_kws={"shrink":.7},
            annot_kws={"size":7}, ax=ax[0])
ax[0].set_title("Spearman correlation", fontsize=9.5)

rec = df.dropna(subset=["doRecommend"]).groupby("rating")["doRecommend"].mean()*100
ax[1].bar(rec.index, rec.values, color=TEAL)
for x, v in rec.items(): ax[1].text(x, v+1.5, f"{v:.0f}", ha="center", fontsize=8)
ax[1].set_ylim(0, 108); ax[1].set_title("Recommend rate by rating (%)", fontsize=9.5)
ax[1].set_xlabel("stars")

med = df.groupby("rating")["words"].median()
ax[2].plot(med.index, med.values, marker="o", color=RED, lw=2)
ax[2].set_title("Median review length by rating", fontsize=9.5)
ax[2].set_xlabel("stars"); ax[2].set_ylabel("words")
plt.tight_layout()

print("recommend rate by rating (%)"); print(rec.round(1).to_string())
print("\nmedian words by rating"); print(med.to_string())
print(f"\n1-star reviews are {med[1]/med[5]:.1f}x longer than 5-star reviews.")

**Figure caption.** Correlation structure and the two relationships that survive cleaning: recommendation rate rises monotonically with rating, and review length falls as rating rises. Both hold on the merged, deduplicated dataset.

## Figure 10 — before-and-after quality dashboard

The brief's conclusion asks how EDA improved the dataset for further analysis. This
dashboard answers that quantitatively rather than in prose.

In [ ]:
fig = plt.figure(figsize=(13.4, 7.2))
gs = fig.add_gridspec(3, 3, hspace=.62, wspace=.30, height_ratios=[.5, 1, 1])

kpi = fig.add_subplot(gs[0, :]); kpi.axis("off")
vals = [("Raw rows (2 sources)", f"{len(src_a)+len(src_b):,}"),
        ("Analytical rows", f"{len(df):,}"),
        ("Duplicates removed", f"{log[0]['rows']+log[1]['rows']:,}"),
        ("Columns kept", f"{len(keep)}"),
        ("Products", f"{df['asin_key'].nunique()}"),
        ("Years covered", f"{df['reviews.date'].dt.year.nunique()}")]
for i, (l, v) in enumerate(vals):
    x = i/len(vals) + .006
    kpi.text(x, .60, v, fontsize=16, fontweight="bold", color="#16202a", transform=kpi.transAxes)
    kpi.text(x, .20, l.upper(), fontsize=7, color="#5a646e", transform=kpi.transAxes)
kpi.set_title("From two raw extracts to one analytical dataset", loc="left",
              fontsize=13, fontweight="bold", pad=4)

a = fig.add_subplot(gs[1, 0])
a.bar(["Source A", "Source B", "Union", "Cleaned"],
      [len(src_a), len(src_b), n_start, len(df)], color=[GREY, GREY, BLUE, TEAL])
a.set_title("Rows through the pipeline", fontsize=9.5); a.tick_params(labelsize=7)

a = fig.add_subplot(gs[1, 1])
before = 100 - (src_b.isna().mean().mean()*100)
after  = 100 - (df.isna().mean().mean()*100)
a.bar(["raw Source B", "cleaned"], [before, after], color=[OCHRE, TEAL])
a.set_ylim(0, 105); a.set_title("Mean column completeness (%)", fontsize=9.5)
for i, v in enumerate([before, after]): a.text(i, v+2, f"{v:.0f}%", ha="center", fontsize=8)
a.tick_params(labelsize=7)

a = fig.add_subplot(gs[1, 2])
a.bar(["raw", "log1p"], [h.skew(), np.log1p(h).skew()], color=[RED, TEAL])
a.set_title("Skew of helpful votes", fontsize=9.5); a.tick_params(labelsize=7)
for i, v in enumerate([h.skew(), np.log1p(h).skew()]): a.text(i, v+1.5, f"{v:.1f}", ha="center", fontsize=8)

a = fig.add_subplot(gs[2, 0])
vc = df["rating"].value_counts(normalize=True).sort_index()*100
a.bar(vc.index, vc.values, color=[RED, RED, OCHRE, TEAL, TEAL])
a.set_title("Rating mix (%)", fontsize=9.5); a.tick_params(labelsize=7)

a = fig.add_subplot(gs[2, 1])
a.bar(rec.index, rec.values, color=PURPLE)
a.set_ylim(0, 108); a.set_title("Recommend rate by rating (%)", fontsize=9.5); a.tick_params(labelsize=7)

a = fig.add_subplot(gs[2, 2])
sm = df["source"].value_counts()
a.bar(sm.index, sm.values, color=[BLUE, OCHRE])
a.set_title("Surviving rows by source", fontsize=9.5); a.tick_params(labelsize=7)
for i, v in enumerate(sm.values): a.text(i, v+600, f"{v:,}", ha="center", fontsize=8)

plt.tight_layout()
print(f"raw union {n_start:,} -> cleaned {len(df):,} ({len(df)/n_start*100:.1f}% retained)")
print(f"mean column completeness: {before:.1f}% -> {after:.1f}%")
print(f"skew of helpful votes: {h.skew():.1f} -> {np.log1p(h).skew():.1f}")

**Figure caption.** Summary dashboard contrasting the raw union with the cleaned dataset: rows retained, completeness recovered, duplicate removal, skew reduction from transformation, and the relationships that survived. This is the one-page answer to 'what did EDA do for this dataset?'

---

# 11. Hypotheses and Conclusion

## Hypotheses generated by this work

The grading scheme allocates 10% to hypothesis generation. A hypothesis names a
mechanism, predicts a direction and states what would falsify it. These four come out of the
cleaned dataset and are testable with data the retailer already holds.

In [ ]:
H = [
 dict(id="H1",
   claim="Customers who rate 1-2 stars write substantially longer reviews because they are "
         "explaining a specific failure, not expressing a general impression.",
   direction=f"median length falls monotonically with rating ({med[1]:.0f} -> {med[5]:.0f} words)",
   test="Regress log(words) on rating with product fixed effects",
   falsified="The gradient disappears once product and category are controlled for"),
 dict(id="H2",
   claim="Negative reviews accumulate more helpful votes because prospective buyers actively "
         "seek warnings before purchase.",
   direction="mean helpful votes highest at 1 star under every outlier policy",
   test="Compare helpful votes per view, not per review, to remove exposure effects",
   falsified="The effect vanishes when normalised by how long a review has been visible"),
 dict(id="H3",
   claim="The recommendation flag is redundant given the star rating and can be dropped from "
         "downstream models without loss.",
   direction=f"recommend rate rises monotonically from {rec.min():.0f}% to {rec.max():.0f}%",
   test="Compare model performance with and without the flag on held-out data",
   falsified="Adding the flag improves out-of-sample performance materially"),
 dict(id="H4",
   claim="Apparent month-on-month movement in this dataset reflects when data was collected "
         "rather than any change in customer sentiment.",
   direction=f"one month holds {monthly['reviews'].max()/len(df)*100:.0f}% of all reviews",
   test="Obtain collection timestamps and model volume against them",
   falsified="Volume tracks a plausible commercial calendar independent of collection dates"),
]
for x in H:
    print(f"[{x['id']}] {x['claim']}")
    print(f"      direction   : {x['direction']}")
    print(f"      how to test : {x['test']}")
    print(f"      falsified if: {x['falsified']}\n")

## Conclusion — what EDA did for this dataset


#### What was done

Two independently-collected extracts totalling **62,992 rows** were profiled, unioned on their 18
shared columns, deduplicated, standardised, type-corrected and enriched — then preprocessed across
all four data kinds the brief names: numeric, textual, time-series and categorical.

#### What it cost, and what it bought

| | Before | After |
|---|---|---|
| Rows | 62,992 across two incompatible files | one analytical table |
| Columns | 21 and 24, overlapping | 18 shared, plus provenance |
| Duplicated reviews | present and uncounted | removed, with the count recorded |
| Dates | strings, unsortable | parsed, resampled to monthly |
| Helpful votes | skew above 60 | log-transformed |
| Text | unusable free text | TF-IDF matrix |
| Missing values | undiagnosed | mechanism identified as MAR given source |

#### The five findings that matter

1. **Integration was the expensive step, not cleaning.** Removing reviews that appeared in *both*
   extracts was by far the largest single reduction. Two sources are not twice the data.
2. **Missingness was explained by provenance.** The helpful-vote and recommendation fields are
   absent in a pattern that tracks which extract a row came from — MAR given source, which makes
   imputation defensible but source-dependent.
3. **Scaling and transformation are different operations.** Every scaler left the skew of helpful
   votes untouched; only the log transform addressed it. Transform first, then scale.
4. **TF-IDF separated what raw counts could not.** Raw frequency returned nearly the same generic
   vocabulary for positive and negative reviews; TF-IDF pulled them apart.
5. **The headline finding is robust.** "Lower ratings attract more helpful votes and longer text"
   holds under raw, winsorised and log-transformed treatments alike — which is what licenses
   reporting it.

#### How the dataset is now better for further analysis

It has one row per unique review with stated provenance; parsed timestamps that support resampling;
numeric columns whose skew has been addressed rather than ignored; a vectorised text representation;
an explicit, documented position on every missing value and every outlier; and a cleaning log that
lets another analyst reproduce or challenge each decision.

#### What it still cannot support

Time trends remain unreliable because collection was not uniform. Neither source is authoritative,
so where they disagreed a precedence rule was applied rather than a truth established. Verified-
purchase analysis is impossible — the field is empty in both extracts. And the reviewers here are
people who chose to write, about a narrow set of products, on one retailer: they are not customers
in general, and no finding should be stated as though they were.
